# 5. Bring custom models

Foundry Local allows you to bring in various models registered in Hugging Face, as well as your own custom models that you've fine-tuned.<br>
In this lesson, we explore how to use custom models in Foundry Local.

In this example, we use "[SmolLM2-Instruct-135M-Humanity-RLHF](https://huggingface.co/tsmatz/SmolLM2-Instruct-135M-Humanity-RLHF)", the model which I have finetuned [SmolLM2-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) with [humanity dataset](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset). This model always responds cheerful answers for any questions. (It's a very small model and you can easily run on CPU.)

## Generate ONNX model

Foundry Local accepts only ONNX models (hardware-accelerated models).<br>
In order to use your own model on Foundry Local, you should then convert your model into ONNX format.

> Note : Foundry Local use ```onnxruntime-genai-core``` at the bottom, and you should take care for the supported model types.<br>
> See [here](https://github.com/microsoft/onnxruntime-genai) for the supported models in ```onnxruntime-genai```.

First, let's check ```onnxruntime-genai-core``` and ```onnxruntime``` version used in your Foundry Local by running the following command.

```
pip freeze | findstr onnx
```

In order to avoid conflicts with other libraries, we will now create a new virtual environment and work there. (Run the following commands.)

```
python -m venv myenv
myenv\Scripts\activate
```

Now let's install the packages required to convert to ONNX format as follows.

As I have mentioned above, take care for the version of libraries, depending on your Foundry Local environment.<br>
In my case, my environment uses NVIDIA GPU and CUDA, so I have installed PyTorch for CUDA as follows, but please see [this document](https://pytorch.org/get-started/locally/) for your appropriate PyTorch installation command.

> Note : When you create ONNX models for GPU execution providers, please also install ```onnxruntime-gpu```. (In this example, we don't need this package.)

```
pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
pip install transformers==5.4.0
pip install onnxruntime==1.25.0 onnx==1.21.0 onnx-ir==0.2.0 onnxscript==0.6.2
pip install onnxruntime-genai==0.13.1
```

You need Hugging Face account. In order to download model from Hugging Face, please login to Hugging Face by running the following command.<br>
(The following ```hf``` command is installed by above ```transformers``` installation.)

```
hf auth login
```

By running the following commands, now let's create your model folder and go to there.

```
mkdir "models/tsmatz/smollm2-instruct-135m-humanity-rlhf-cpu-1"
cd "models/tsmatz/smollm2-instruct-135m-humanity-rlhf-cpu-1"
```

In this example, we use "[SmolLM2-Instruct-135M-Humanity-RLHF](https://huggingface.co/tsmatz/SmolLM2-Instruct-135M-Humanity-RLHF)", the model which I have finetuned [SmolLM2-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) with [humanity dataset](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset).

By running the following command, let's build ONNX format of this model.

For the purpose of demonstration, we create ONNX model optimized for CPU in this example.<br>
But in production environment, please create a model depending on your device acceleration, such as "```webgpu```", etc. (CPU model can only use int quantization, which can **reduce accuracy** in your model.)

```
python -m onnxruntime_genai.models.builder -m tsmatz/SmolLM2-Instruct-135M-Humanity-RLHF -p int4 -e cpu -o v1
```

Now you can quit virtual environment.

```
deactivate
```
Your ONNX model is saved on ```./models/tsmatz/smollm2-instruct-135m-humanity-rlhf-cpu-1/v1```.

In [1]:
!dir models\tsmatz\smollm2-instruct-135m-humanity-rlhf-cpu-1\v1

 Volume in drive C is Windows
 Volume Serial Number is BA8A-5909

 Directory of c:\Demo\foundry-local\models\tsmatz\smollm2-instruct-135m-humanity-rlhf-cpu-1\v1

04/24/2026  09:20 PM    <DIR>          .
04/24/2026  09:20 PM    <DIR>          ..
04/24/2026  09:20 PM               373 chat_template.jinja
04/24/2026  09:20 PM             1,503 genai_config.json
04/24/2026  09:20 PM           223,477 model.onnx
04/24/2026  09:20 PM       199,557,120 model.onnx.data
04/24/2026  09:20 PM         3,522,871 tokenizer.json
04/24/2026  09:20 PM               368 tokenizer_config.json
               6 File(s)    203,305,712 bytes
               2 Dir(s)  30,168,875,008 bytes free


## Generate chat template

The chat model uses various chat prompt formats. You need to create ```inference_model.json``` file to tell Foundry Local which chat prompt format this model uses.

In this example, standard OpenAI chat template format (ChatML) is used, and we then create the following definition as ```inference_model.json``` file.

> Note : By using ```tokenizer.apply_chat_template()``` method in Hugging Face API, you can retreive prompt template from tokenizer that model uses.

In [2]:
%%writefile models/tsmatz/smollm2-instruct-135m-humanity-rlhf-cpu-1/v1/inference_model.json
{
  "Name": "smollm2-instruct-135m-humanity-rlhf-cpu:1",
  "PromptTemplate": {
    "system": "\u003C|im_start|\u003Esystem\n{Content}\u003C|im_end|\u003E",
    "user": "\u003C|im_start|\u003Euser\n{Content}\u003C|im_end|\u003E",
    "assistant": "\u003C|im_start|\u003Eassistant\n{Content}\u003C|im_end|\u003E",
    "prompt": "\u003C|im_start|\u003Euser\n{Content}\u003C|im_end|\u003E\n\u003C|im_start|\u003Eassistant"
  }
}

Writing models/tsmatz/smollm2-instruct-135m-humanity-rlhf-cpu-1/v1/inference_model.json


## Run your own model in Foundry Local

Now let's invoke your own ONNX model.

To use your own generated model, set the folder containing your model as ```model_cache_dir``` property in Foundry Local manager.

In [3]:
from foundry_local_sdk import Configuration, FoundryLocalManager

# initialize manager and execution providers (see Lesson1)
config = Configuration(
    app_name="foundry_local_samples",
    model_cache_dir="./models",
)
FoundryLocalManager.initialize(config)
manager = FoundryLocalManager.instance
result = manager.download_and_register_eps()

When you list available cached models, you can see the model you have created above.

In [4]:
cached_models = manager.catalog.get_cached_models()
for m in cached_models:
    print(m.id)

smollm2-instruct-135m-humanity-rlhf-cpu:1


Pick up and invoke this cached model as follows.

In [5]:
import json

model = [m for m in cached_models if "smollm2-instruct-135m-humanity" in m.id][0]
model.load()

client = model.get_chat_client()
completion = client.complete_chat([
    {
        "role": "user",
        "content": "What is the best gift to give a friend who loves the outdoors?"
    }
])

print(json.dumps(completion.choices[0].delta, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': ')))

{
    "content": "That's a classic gift, and the challenge lies in the little details that can make or break a friend's special day.\n\nWhen it comes to a gift to celebrate an outdoor or sporting occasion, you can be very specific and specific it is.\n\nIt would be a huge help to provide a specific weather forecast for the day in question, so you can plan accordingly.\n\nIt can also be extremely difficult to ensure a little bit of an adjustment to a friend's special day can be put together to a very high standard.\n\nIt's much the same as it is for a friend to celebrate a special occasion by putting together a little bit of an adjustment to a friend's special day can be put together to a very high standard.",
    "role": "assistant",
    "tool_calls": []
}


## Unload model

In [6]:
model.unload()